In [30]:
import pandas as pd

In [31]:
df = pd.read_csv(f'../dataset/dataset-tickets-multi-lang-4-20k.csv', encoding='utf-8')

In [32]:
df = df[df['language'] == 'en']

In [33]:
df.info()

<class 'pandas.DataFrame'>
Index: 11923 entries, 1 to 19997
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   subject   10891 non-null  str  
 1   body      11922 non-null  str  
 2   answer    11920 non-null  str  
 3   type      11923 non-null  str  
 4   queue     11923 non-null  str  
 5   priority  11923 non-null  str  
 6   language  11923 non-null  str  
 7   tag_1     11923 non-null  str  
 8   tag_2     11913 non-null  str  
 9   tag_3     11879 non-null  str  
 10  tag_4     11050 non-null  str  
 11  tag_5     7885 non-null   str  
 12  tag_6     4538 non-null   str  
 13  tag_7     2565 non-null   str  
 14  tag_8     1327 non-null   str  
dtypes: str(15)
memory usage: 1.5 MB


In [34]:
df = df.dropna(subset=['body']).copy()

In [35]:
df[['subject', 'body']]

,subject,body
1,Customer Support Inquiry,Seeking information on digital strategies that...
2,Data Analytics for Investment,I am contacting you to request information on ...
4,Security,"Dear Customer Support, I am reaching out to in..."
5,Concerns About Securing Medical Data on 2-in-1...,Inquiring about best practices for securing me...
7,Problem with Integration,"The integration stopped working unexpectedly, ..."
...,...,...
19992,Guidelines for Securing Medical Data in OBS St...,Seeking details on securing medical data using...
19993,NaN,Can you provide information on digital strateg...
19994,Support for Marketing Enhancements,Request for assistance in improving digital ma...
19995,Assistance Needed for IFTTT Docker Integration,I am facing integration problems with IFTTT Do...


In [36]:
df['subject'] = df['subject'].fillna('')

In [37]:
df['Subject + body'] = df['subject'] + ' ' + df['body']

In [38]:
df['Subject + body']

1        Customer Support Inquiry Seeking information o...
2        Data Analytics for Investment I am contacting ...
4        Security Dear Customer Support, I am reaching ...
5        Concerns About Securing Medical Data on 2-in-1...
7        Problem with Integration The integration stopp...
                               ...                        
19992    Guidelines for Securing Medical Data in OBS St...
19993     Can you provide information on digital strate...
19994    Support for Marketing Enhancements Request for...
19995    Assistance Needed for IFTTT Docker Integration...
19997     Hello Customer Support, I am inquiring about ...
Name: Subject + body, Length: 11922, dtype: str

In [39]:
from sklearn.model_selection import train_test_split

In [40]:
X = df['Subject + body']
y = df['queue']

In [41]:
X

1        Customer Support Inquiry Seeking information o...
2        Data Analytics for Investment I am contacting ...
4        Security Dear Customer Support, I am reaching ...
5        Concerns About Securing Medical Data on 2-in-1...
7        Problem with Integration The integration stopp...
                               ...                        
19992    Guidelines for Securing Medical Data in OBS St...
19993     Can you provide information on digital strate...
19994    Support for Marketing Enhancements Request for...
19995    Assistance Needed for IFTTT Docker Integration...
19997     Hello Customer Support, I am inquiring about ...
Name: Subject + body, Length: 11922, dtype: str

In [42]:
X.shape

(11922,)

In [44]:
y.shape

(11922,)

In [45]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [47]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
vectorizer.get_feature_names_out()

array(['00', '000', '04', ..., 'zurückzuführen', 'zusammenbruch',
       'überprüft'], shape=(5014,), dtype=object)

In [48]:
X_train.shape

(9537,)

In [49]:
X_test.shape

(2385,)

In [57]:
X_train_tfidf.shape

(9537, 5014)

In [58]:
X_test_tfidf.shape

(2385, 5014)

In [50]:
len(vectorizer.get_feature_names_out())

5014

In [51]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42, max_iter=5000).fit(X_train_tfidf, y_train)


In [52]:
model.predict(X_test_tfidf[:5])

array(['Technical Support', 'Technical Support', 'Product Support',
       'Technical Support', 'Technical Support'], dtype=object)

In [53]:
model.predict_proba(X_test_tfidf[:5])

array([[0.03800995, 0.10484236, 0.00522471, 0.02066237, 0.12779164,
        0.17935588, 0.0308845 , 0.01129518, 0.00881506, 0.47311834],
       [0.00489872, 0.06658304, 0.00399788, 0.0078815 , 0.15330211,
        0.04959663, 0.00574259, 0.00811135, 0.01154973, 0.68833644],
       [0.02697818, 0.11540943, 0.0054827 , 0.01010344, 0.07619139,
        0.29558452, 0.03301069, 0.02171531, 0.1212117 , 0.29431264],
       [0.02683554, 0.10553098, 0.01314641, 0.02086099, 0.16830814,
        0.13710914, 0.08722987, 0.01968623, 0.0311914 , 0.3901013 ],
       [0.10628267, 0.1437542 , 0.00624596, 0.02137198, 0.14705634,
        0.0683406 , 0.03646489, 0.00961607, 0.01032927, 0.45053803]])

In [54]:
majority_class = y_train.mode().iloc[0]
majority_class_baseline = (y_test == majority_class).mean()
train_accuracy = model.score(X_train_tfidf, y_train)
test_accuracy = model.score(X_test_tfidf, y_test)

print(f'majority class from train = {majority_class}')
print(f'majority-class baseline accuracy = {majority_class_baseline:.6f}')
print(f'train accuracy = {train_accuracy:.6f}')
print(f'test accuracy = {test_accuracy:.6f}')

majority class from train = Technical Support
majority-class baseline accuracy = 0.286373
train accuracy = 0.581525
test accuracy = 0.436059


In [55]:
noise = 'zurückzuführen'
df[df['Subject + body'].str.contains(noise, case=False, na=False)]
english_sample = df.sample(n=min(20, len(df)), random_state=2026)[['language', 'subject', 'body', 'queue']]
english_sample.to_string(index=True, max_colwidth=300)

"      language                                                                     subject                                                                                                                                                                                                                                                                                                         body                  queue\n4162        en                                         Medical Data Encryption Malfunction                           An unforeseen medical data encryption malfunction seems to be due to a possibly outdated SAP ERP module. I have already restarted the Speicherkartenleser and PyTorch services, yet the problem continues. I would greatly appreciate your help in addressing this issue promptly.      Technical Support\n4301        en                               Clarification on SaaS Project Management Fees  I am keen to learn more about the SaaS project management solution, particu

In [56]:
test_predictions = pd.Series(model.predict(X_test_tfidf), index=y_test.index, name='predicted queue')
error_analysis = pd.DataFrame({
    'text': X_test,
    'actual queue': y_test,
    'predicted queue': test_predictions,
})
error_analysis = error_analysis[error_analysis['actual queue'] != error_analysis['predicted queue']]
error_sample = error_analysis.sample(n=min(10, len(error_analysis)), random_state=2026)
error_sample.to_string(index=False, max_colwidth=500)

'                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                text        actual queue   predicted queue\n                                                                                                                                                                                                                                                                                                     Improve Data Visualization in KNIME I am in need of assistance to optimize investment strategies and portfolio performance analysis using KNIME. Could you offer some 

In [63]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_train_accuracy = nb_model.score(X_train_tfidf, y_train)
nb_test_accuracy = nb_model.score(X_test_tfidf, y_test)

comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Multinomial Naive Bayes'],
    'Train Accuracy': [train_accuracy, nb_train_accuracy],
    'Test Accuracy': [test_accuracy, nb_test_accuracy],
})
comparison

,Model,Train Accuracy,Test Accuracy
0,Logistic Regression,0.581525,0.436059
1,Multinomial Naive Bayes,0.425501,0.387421


In [64]:
from sklearn.metrics import classification_report, confusion_matrix

models = {
    'Logistic Regression': model,
    'Multinomial Naive Bayes': nb_model,
}

for model_name, classifier in models.items():
    predictions = classifier.predict(X_test_tfidf)
    labels = classifier.classes_
    report = classification_report(y_test, predictions, zero_division=0)
    matrix = confusion_matrix(y_test, predictions, labels=labels)
    report_dict = classification_report(
        y_test,
        predictions,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )

    pair_counts = {}
    for row_index, actual_label in enumerate(labels):
        for column_index, predicted_label in enumerate(labels):
            if row_index != column_index and matrix[row_index, column_index] > 0:
                pair = tuple(sorted((actual_label, predicted_label)))
                pair_counts[pair] = pair_counts.get(pair, 0) + matrix[row_index, column_index]
    most_confused_pair = max(pair_counts, key=pair_counts.get)

    print(f'=== {model_name} ===')
    print(report)
    print('Confusion matrix:')
    print(pd.DataFrame(matrix, index=labels, columns=labels))
    print(f"Lowest recall: {min(labels, key=lambda label: report_dict[label]['recall'])}")
    print(f"Lowest precision: {min(labels, key=lambda label: report_dict[label]['precision'])}")
    print(f'Most confused pair: {most_confused_pair} ({pair_counts[most_confused_pair]} errors)')
    print(f"Macro F1: {report_dict['macro avg']['f1-score']:.6f}")
    print(f"Weighted F1: {report_dict['weighted avg']['f1-score']:.6f}")
    print()

=== Logistic Regression ===
                                 precision    recall  f1-score   support

           Billing and Payments       0.79      0.69      0.74       261
               Customer Service       0.31      0.32      0.32       372
                General Inquiry       0.00      0.00      0.00        34
                Human Resources       1.00      0.15      0.26        41
                     IT Support       0.38      0.14      0.21       278
                Product Support       0.38      0.37      0.38       446
          Returns and Exchanges       0.88      0.06      0.11       116
            Sales and Pre-Sales       0.67      0.06      0.11        66
Service Outages and Maintenance       0.91      0.34      0.50        88
              Technical Support       0.41      0.71      0.52       683

                       accuracy                           0.44      2385
                      macro avg       0.57      0.28      0.31      2385
                   we